# PCA - Reducción Dimensional
Componentes principales sobre defunciones de Buenos Aires

## Setup y carga de datos limpios

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from etl.main import run as run_etl

df = run_etl('../data/raw/defunciones-ocurridas-y-registradas-en-la-republica-argentina-entre-los-anos-2005-2022.csv')
df.shape

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

## Preparación de datos y clustering base

In [ ]:
df_piModNoSup = df.copy()

ordenEdad = ['De a 0 a 14 anios','De 15 a 34 anios',
            'De 35 a 54 anios','De 55 a 74 anios',
            'De 75 anios y mas']
ordEdad = OrdinalEncoder(categories=[ordenEdad])
df_piModNoSup['grupo_edad'] = ordEdad.fit_transform(df_piModNoSup[['grupo_edad']])

X_real = df_piModNoSup[['grupo_edad', 'cantidad', 'anio']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_real)

kmeans_final = KMeans(n_clusters=4, n_init='auto', random_state=42)
clusters_real = kmeans_final.fit_predict(X_scaled)

## PCA y loadings

In [ ]:
# 1. Reducimos los datos a 2 dimensiones para poder graficar
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

# Crear un DataFrame para ver los pesos/cargas de cada variable en los componentes
loadings = pd.DataFrame(
    pca.components_.T,  # Toma el "peso" de cada valor(importancia), y con T, quedan como filas y los ejes como columnas
    columns=['Componente_1 (Eje X)', 'Componente_2 (Eje Y)'], # Ejes
    index=X_real.columns                  # Trae los nombres de las variables
)

print("--- REBORDES Y PESOS DEL PCA ---")
print(loadings.round(3)) # Redondeamos a 3 decimales para que sea fácil de leer
# Util para saber a que eje pertenece cada variable
# La variable mas cercana a 1 o -1, muestra que esta mayormente representada en ese eje

## Visualización de clusters en plano PCA

In [ ]:
X_pca = pca.fit_transform(X_scaled)
# 2. Visualización
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_real, cmap='viridis')
plt.title("Visualización de Clusters con PCA")
plt.xlabel("Distancia del promedio de cantidad y edad")
plt.ylabel("Distancia del promedio de año")
plt.show()

## Validación: distribución por causa de defunción

In [ ]:
# Creámos un DataFrame temporal con las coordenadas del PCA
df_visualizacion = pd.DataFrame(X_pca, columns=['Componente_1', 'Componente_2'])

# Le pegamos la columna de texto que NO usamos en el modelo
df_visualizacion['supracategoria'] = df['supracategoria'].values

# Graficamos usando Seaborn para ver si los colores se agrupan
plt.figure(figsize=(12, 7))

sns.scatterplot(
    data=df_visualizacion,
    x='Componente_1',
    y='Componente_2',
    hue='supracategoria', # El color muestra la enfermedad
    alpha=0.6,
    palette='tab20' # Una paleta con muchos colores para distinguir bien las causas
)

plt.title("Validación del PCA: Distribución por Causa de Defunción")
plt.xlabel("Distancia del promedio de cantidad y edad")
plt.ylabel("Distancia del promedio de año")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') # Mueve la leyenda afuera para que no tape
plt.tight_layout()
plt.show()

## Segmentación final con centroides en plano PCA

In [ ]:
# 3. ¡EL TRUCO MAESTRO!: Transformamos también los centroides al espacio PCA
# Los centroides originales tienen 3 coordenadas. Con esto los bajamos a 2 coordenadas.
centroides_pca = pca.transform(kmeans_final.cluster_centers_)

# Configuración del gráfico
plt.figure(figsize=(11, 7))

# Nombres personalizados
nombres_grupos = [
    "Mortalidad Moderada Alta / Edades Avanzadas",  # Cluster 0
    "Casos Extremos",   # Cluster 1
    "Baja Mortalidad / Poblacion Joven",  # Cluster 2
    "Mortalidad Moderada / Edades Avanzadas"   # Cluster 3
]

# Graficar con coordenadas de X_pca (0 = Eje X, 1 = Eje Y)
for i in range(4):
    plt.scatter(X_pca[clusters_real == i, 0],
                X_pca[clusters_real == i, 1],
                s=50,
                label=nombres_grupos[i],
                alpha=0.6)

# Graficamos los centroides usando sus nuevas coordenadas en el plano PCA
plt.scatter(centroides_pca[:, 0], centroides_pca[:, 1],
            s=250, c='black', marker='X', edgecolors='white', label='Centroides')

# Títulos y etiquetas reales basados en tu matriz de cargas (Loadings)
plt.title('Segmentación de Defunciones con K-Means (K=4) mediante PCA', fontsize=14, pad=15)
plt.xlabel("Distancia del promedio de cantidad y edad")
plt.ylabel("Distancia del promedio de año")

plt.axhline(0, color='gray', linestyle='--', alpha=0.5) # Línea guía del centro
plt.axvline(0, color='gray', linestyle='--', alpha=0.5) # Línea guía del centro

plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()